# Ordering


## Warning: This notebook will place live orders

Use a paper trading account (during market hours).


In [1]:
from functools import partial
import ipywidgets as widgets
from IPython.display import DisplayHandle, display, clear_output, update_display, HTML, display_html
import time
import pandas as pd
import datetime, traceback, logging


In [2]:
# to import local code
# https://stackoverflow.com/questions/61058798/python-relative-import-in-jupyter-notebook
# https://stackoverflow.com/questions/34478398/import-local-function-from-a-module-housed-in-another-directory-with-relative-im

import os, sys
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    # sys.path.append(parent_dir)
    sys.path.insert(0, parent_dir)
    parent_dir


In [3]:
# import pytz
# local_tz = pytz.timezone('US/Eastern')  # Adjust for your local timezone, America/New_York

import ib_insync
ib_insync.ib.install_custom_repr_()

# def install_custom_repr_():
#     # monkey patch ib_insync.objects.TradeLogEntry.__repr__ to use friendlier time format
#     def trade_log_entry_repr(self):
#         return f"TradeLogEntry(time={self.time.astimezone(local_tz).strftime('%H:%M:%S.%f')}" \
#             + (f", status='{self.status}'") \
#             + (f", message='{self.message}'" if self.message else '') \
#             + (f", errorCode={self.errorCode})" if self.errorCode else '') \
#             + ")"
#     ib_insync.objects.TradeLogEntry.__repr__orig = ib_insync.objects.TradeLogEntry.__repr__
#     ib_insync.objects.TradeLogEntry.__repr__ = trade_log_entry_repr

# install_custom_repr_()

In [4]:
from ib_insync import *
util.startLoop()
ib = IB()

In [ ]:
# util.logToConsole()
def apiStartEvent():
    print("API Start Event")

def apiEndEvent():
    print("API End Event")

def apiErrorEvent(errorMsg: str):
    print(f"API Error {errorMsg}")

def throttleStartEvent():
    print("Throttle Start Event")

def throttleEndEvent():
    print("Throttle End Event")

def onConnectedEvent():
    print("Connected Event")

def onDisconnectedEvent():
    print("Disconnected Event")

def onNewOrderEvent(trade: Trade):
    print(f"New Order Event {trade}")

def onOpenOrderEvent(trade: Trade):
    print(f"Open Order Event {trade}")

def onOrderStatusEvent(trade: Trade):
    print(f"Order Status Event {trade}")

def onExecDetailsEvent(trade: Trade, fill: Fill):
    print(f"Exec Details Event {trade} {fill}")

def onErrorEvent(reqId, errorCode, errorString, contract):
    print(f"Error Event {reqId} {errorCode} {errorString} {contract}")

def onTimeoutEvent(idlePeriod: float):
    print(f"Timeout Event {idlePeriod}")


In [5]:
def generic_event_handler(event_name: str, *args, **kwargs):
    print(f"{event_name}: {args}" + (f"{kwargs}" if kwargs else ''))

In [6]:

# ib.connectedEvent += print # onConnectedEvent
# ib.disconnectedEvent += print # onDisconnectedEvent
ib.newOrderEvent += partial(generic_event_handler, 'newOrderEvent')
ib.openOrderEvent += partial(generic_event_handler, 'openOrderEvent')
ib.orderStatusEvent += partial(generic_event_handler, 'orderStatusEvent')
ib.cancelOrderEvent += partial(generic_event_handler, 'cancelOrderEvent')
ib.execDetailsEvent += partial(generic_event_handler, 'execDetailsEvent')
ib.errorEvent += partial(generic_event_handler, 'errorEvent')
# ib.timeoutEvent += print # onTimeoutEvent

# ib.client.apiStart.connect(apiStartEvent)
# ib.client.apiEnd.connect(apiEndEvent)
# ib.client.apiError.connect(apiErrorEvent)
# ib.client.throttleStart.connect(throttleStartEvent)
# ib.client.throttleEnd.connect(throttleEndEvent)


In [ ]:
# ib.updatePortfolioEvent += partial(generic_event_handler, 'updatePortfolioEvent')
# ib.positionEvent += partial(generic_event_handler, 'positionEvent')

In [9]:
# ib = IB()
port_2 = 7497
port_1 = 4002
port_3 = 7496
# util.logToConsole(logging.DEBUG)
ib.connect('127.0.0.1', port_2, clientId=1300)
ib.managedAccounts()

['DU9749485']

In [10]:
# util.logToConsole(logging.DEBUG)
ib.portfolio()

[]

In [ ]:
# util.logToConsole(logging.DEBUG)

In [ ]:
ibtrades = [t for t in ib.trades() if t.contract.symbol == 'MBT' and t.orderStatus.status != 'Cancelled']
ibfills = [t.fills for t in ibtrades]
ibexecs = [fill.execution for sublist in ibfills for fill in sublist]

In [ ]:
ibexecs

In [ ]:
util.tree(ibfills[1])

Create a contract and a market order:

https://interactivebrokers.github.io/tws-api/available_orders.html

In [ ]:
contract = Forex('EURUSD')
ib.qualifyContracts(contract)

# buysell = 'BUY'
buysell = 'SELL'
lmtOrder = LimitOrder(buysell, 50000, 1.11)
mktOrder = MarketOrder(buysell, 50000)

In [ ]:
# exchange = 'OVERNIGHT'
exchange = 'SMART'
primaryExchange = 'ARCA'
contract = Stock('SPY', exchange, 'USD')
discretionaryAmt = 1.0


In [ ]:

# mktOrder = MarketOrder(buysell, 100, outsideRth=True)
buysell = 'BUY'
mktOrder = MarketOrder(buysell, 100)
# mktOrder.account = 'DU11314372'
# mktOrder.faGroup = 'DU11314372'
# mktOrder.clearingAccount = 'DU11314372'


In [ ]:
lmtOrder = LimitOrder(buysell, 100, 570.02, discretionaryAmt=discretionaryAmt)

placeOrder will place the order order and return a ``Trade`` object right away (non-blocking):

In [8]:
class TradeManager:
    def __init__(self, trade: Trade):
        self.trade = trade
        self.trade.statusEvent += self.on_status_event
        self.trade.modifyEvent += self.on_modify_event
        self.trade.fillEvent += self.on_fill_event
        self.trade.commissionReportEvent += self.on_commission_report_event
        self.trade.filledEvent += self.on_filled_event
        self.trade.cancelEvent += self.on_cancel_event
        self.trade.cancelledEvent += self.on_cancelled_event

    def on_status_event(self, trade: Trade):
        print(f"Status Event: {trade}")

    def on_modify_event(self, trade: Trade):
        print(f"Modify Event: {trade}")

    def on_fill_event(self, trade: Trade, fill: Fill):
        print(f"Fill Event: {trade}, {fill}")

    def on_commission_report_event(self, trade: Trade, fill: Fill, report: CommissionReport):
        print(f"Commission Report Event: {trade}, {fill}, {report}")

    def on_filled_event(self, trade: Trade):
        print(f"Filled Event: {util.tree(trade)}")

    def on_cancel_event(self, trade: Trade):
        print(f"Cancel Event: {trade}")

    def on_cancelled_event(self, trade: Trade):
        print(f"Cancelled Event: {trade}")



In [ ]:
ib.positions()

In [ ]:
ib.openTrades()

In [ ]:
ib.reqOpenOrders()

In [ ]:
ib.openOrders()

In [ ]:
ib.trades()

In [ ]:
def _execs_list2tuplst(execs: list[Execution]) -> list[tuple]:
    return [(exec_.execId, exec_.time, exec_.side) for exec_ in execs] if execs else []


In [ ]:
ibfills

In [ ]:
_execs_list2tuplst(ibexecs)

In [ ]:
sorted(list(ibexecs), key=lambda x: x[1], reverse=True)[0]

In [ ]:
# get all trades so far
trades = [t for t in ib.trades() if t.contract.symbol == 'SPY']
trades

In [ ]:
# get the most recent trade (whose permId is highest)
most_recent_trade = max(trades, key=lambda t: t.fills)
most_recent_trade

In [ ]:
permIds = [(o.orderId, o.permId) for o in [t.order for t in ib.trades() if t.contract.symbol == 'SPY']]
permIds

In [ ]:
permIds

In [ ]:
orderId = 163
permId = None
for item in permIds:
  print(item, orderId)
  match item:
    case (orderId, x):
        permId = x
        print(f"orderId: {orderId}, permId: {permId}")
        break
else:
    print("Not found")

In [ ]:
# Your list of tuples
data = [(0, 1355514819),
        (0, 1355514823),
        (0, 1355514775),
        (0, 1355514786),
        (0, 1355514789),
        (0, 1355514813),
        (0, 1355514803),
        (0, 1355514800),
        (0, 1355514807),
        (158, 1355514902),
        (159, 1355514903),
        (160, 1355514904),
        (161, 1355514905),
        (162, 1355514906),
        (163, 1355514907),
        (164, 1355514908)]

# Your orderId variable
want = 160

# Initialize permId to None
permId = next((x for orderId, x in data if orderId == want), None)

# # Iterate through the list of tuples
# for item in data:
#     # Unpack the tuple
#     orderId, x = item

#     # Check if the orderId matches
#     if orderId == want:
#         # Assign the permId
#         permId = x

#         # Print the result
#         print(f"orderId: {orderId}, permId: {permId}")

#         # Break out of the loop
#         break
# Output the result
print(f"permId: {permId}, orderId: {orderId}")


In [ ]:
# Create an instance of TradeManager with the existing trade object
trade_manager = TradeManager(trade := ib.placeOrder(contract, mktOrder))

In [ ]:
excFilt = ExecutionFilter(symbol='MES')
all_fills = await ib.reqExecutionsAsync(excFilt)
all_execs = [fill.execution for fill in all_fills]

In [ ]:
def execs_list2tuplst(execs: list[Execution]):
    return [(exec_.execId, exec_.time, exec_.side) for exec_ in execs] if execs else []


In [ ]:
a=set(execs_list2tuplst(all_execs))

In [ ]:
ibtrades = [t for t in ib.trades() if t.contract.symbol == 'MES']
ibfills = [t.fills for t in ibtrades]
ibexecs = [fill.execution for sublist in ibfills for fill in sublist]

In [ ]:
b=set(execs_list2tuplst(ibexecs))

In [ ]:
a.intersection(b)

In [ ]:
def onPositionUpdate(newpos):
    print(f"onPositionUpdate: {datetime.datetime.now().isoformat(' ')} {newpos} id={id(newpos)}")
    print(ib.trades())
    # traceback.print_stack()

# util.logToConsole(logging.DEBUG)
# ib.positionEvent.connect(onPositionUpdate, error=lambda x: print(f"error: {x}"), done=lambda: print("done"))
ib.positionEvent += onPositionUpdate

In [ ]:
util.logToConsole(logging.DEBUG)

In [ ]:
trade = ib.placeOrder(contract, lmtOrder)


In [ ]:
trade

In [ ]:
util.tree(trade)

In [ ]:
# get order.permId whose execution.time is the most recent
xf = [f for f in [t.fills for t in [trade for trade in trades]]]

In [ ]:
flattened_xf = [item for sublist in xf for item in sublist]
[(f.execution.time, f.execution.permId, f.execution.execId) for f in flattened_xf]
most_recent_fill = max(flattened_xf, key=lambda f: f.execution.time)
most_recent_fill#.execution.permId

In [ ]:
trade.statusEvent += print
trade.modifyEvent += print
trade.fillEvent += print
trade.commissionReportEvent += print
trade.filledEvent += print
trade.cancelEvent += print
trade.cancelledEvent += print

# print(trade)

In [ ]:
trade

In [ ]:
ib.positions()

In [ ]:
trade.log

In [ ]:
trade.log[0].__repr__()

In [ ]:
import asyncio
import time
import ipywidgets as widgets
from IPython.display import Markdown, display

class OutputLogger:
    def __init__(self):
        self.output_area = widgets.Output()
        display(self.output_area)
        self.buffer = ""
        self.lock = asyncio.Lock()

    def _format_message(self, level, message, elapsed=None):
        prefix = {
            "INFO": "🔹",
            "WARNING": "⚠️",
            "ERROR": "❌"
        }.get(level, "🔸")
        time_str = f" _(Elapsed: {elapsed:.2f}s)_" if elapsed is not None else ""
        
        # Ensure line breaks are rendered properly using Markdown block and double spaces
        safe_message = message.replace('\n', '  \n')
        return f"{prefix} **{level}**: {safe_message}{time_str}  \n"

    async def log_async(self, level, message, timed=False):
        start = time.time() if timed else None

        async with self.lock:
            elapsed = time.time() - start if timed else None
            formatted = self._format_message(level, message, elapsed)
            self.buffer += formatted
            with self.output_area:
                self.output_area.clear_output(wait=True)
                display(Markdown(self.buffer))

    async def info_async(self, message, timed=False):
        await self.log_async("INFO", message, timed)

    async def warn_async(self, message, timed=False):
        await self.log_async("WARNING", message, timed)

    async def error_async(self, message, timed=False):
        await self.log_async("ERROR", message, timed)

    def log(self, level, message, timed=False):
        """Sync wrapper for quick use in non-async context."""
        asyncio.run(self.log_async(level, message, timed))

    def info(self, message, timed=False):
        self.log("INFO", message, timed)

    def warn(self, message, timed=False):
        self.log("WARNING", message, timed)

    def error(self, message, timed=False):
        self.log("ERROR", message, timed)

Create a "buy protect" order

Create instrument

In [12]:
action = 'SELL'
quantity = 1
futsym = 'ES'
exchange = 'CME'
expiry = '202512'
contract = Future(futsym, expiry, exchange)
conDet = ib.reqContractDetails(contract)
mintick = conDet[0].minTick
# discretionaryAmt = 1.0

In [ ]:
# mktOrder = MarketOrder(action, quantity) # , outsideRth=True
# trade = ib.placeOrder(contract, mktOrder)

In [10]:
from ib_insync import MarketOrder, StopOrder

# Submit a market order and capture fills via orderStatusEvent, then submit a stop order at the fill price.

def print_to_output(*args, **kwargs):
    print(*args, **kwargs)
# # Function to handle order status and submit stop order when filled
# def on_order_status(trade):
#     status = trade.orderStatus.status
#     if status == 'Filled':
#         filled_qty = trade.orderStatus.filled
#         avg_fill_price = trade.orderStatus.avgFillPrice
#         print(f"Order filled: qty={filled_qty}, price={avg_fill_price}")

#         # Submit stop order at fill price
#         stop_order = StopOrder('SELL', filled_qty, avg_fill_price)
#         stop_trade = ib.placeOrder(contract, stop_order)
#         print(f"Stop order submitted: {stop_order}")

#         # Remove event handler after fill
#         ib.orderStatusEvent -= on_order_status

def on_order_filled(trade: Trade):
    status = trade.orderStatus.status
    assert status == 'Filled'
    filled_qty = trade.orderStatus.filled
    avg_fill_price = trade.orderStatus.avgFillPrice
    msg = f"Order filled: qty={filled_qty}, price={avg_fill_price}; "

    # Submit stop order at fill price
    reverseAction = 'BUY' if trade.order.action == 'SELL' else 'SELL'
    stopPrice = avg_fill_price - 1.0 if reverseAction == 'SELL' else avg_fill_price + 1.0
    stop_order = StopOrder(reverseAction, filled_qty, stopPrice)
    # stop_order.parentId = trade.order.orderId  # Link stop order to the original trade
    stop_trade: Trade = ib.placeOrder(contract, stop_order)
    stop_trade.statusEvent += partial(generic_event_handler, 'stopTradeStatusEvent')
    stop_trade.fillEvent += partial(generic_event_handler, 'stopTradeFillEvent')
    stop_trade.filledEvent += partial(generic_event_handler, 'stopTradeFilledEvent')
    print_to_output(msg + f"Stop order submitted: {stop_order}")

def on_order_status(trade: Trade):
    status = trade.orderStatus.status
    assert status == 'Filled'
    filled_qty = trade.orderStatus.filled
    avg_fill_price = trade.orderStatus.avgFillPrice
    print_to_output(f"Market order filled: qty={filled_qty}, price={avg_fill_price}")

    # Remove event handler after fill
    trade.statusEvent -= on_order_status


In [ ]:
# # Submit market order
action = 'BUY'
market_order = MarketOrder(action, quantity)
# ib.orderStatusEvent += on_order_status
trade: Trade = ib.placeOrder(contract, market_order)
trade.statusEvent += partial(generic_event_handler, 'tradeStatusEvent')
trade.fillEvent += partial(generic_event_handler, 'tradeFillEvent')
trade.filledEvent += partial(generic_event_handler, 'tradeFilledEvent')
trade.filledEvent += on_order_filled

newOrderEvent: (Trade(contract=Future(symbol='ES', lastTradeDateOrContractMonth='202509', exchange='CME'), order=MarketOrder(orderId=348, clientId=13, action=BUY, totalQuantity=1), orderStatus=OrderStatus(orderId=348, status=PendingSubmit), log=[TradeLogEntry(time=09:59:20,996, status=PendingSubmit)]),)


openOrderEvent: (Trade(contract=Future(symbol='ES', lastTradeDateOrContractMonth='202509', exchange='CME'), order=MarketOrder(orderId=348, clientId=13, permId=2048127267, action=BUY, totalQuantity=1), orderStatus=OrderStatus(orderId=348, status=PendingSubmit), log=[TradeLogEntry(time=09:59:20,996, status=PendingSubmit)]),)
orderStatusEvent: (Trade(contract=Future(symbol='ES', lastTradeDateOrContractMonth='202509', exchange='CME'), order=MarketOrder(orderId=348, clientId=13, permId=2048127267, action=BUY, totalQuantity=1), orderStatus=OrderStatus(orderId=348, permId=2048127267, status=PreSubmitted, remaining=1, clientId=13), log=[TradeLogEntry(time=09:59:20,996, status=PendingSubmit), TradeLogEntry(time=09:59:21,206, status=PreSubmitted)]),)
tradeStatusEvent: (Trade(contract=Future(symbol='ES', lastTradeDateOrContractMonth='202509', exchange='CME'), order=MarketOrder(orderId=348, clientId=13, permId=2048127267, action=BUY, totalQuantity=1), orderStatus=OrderStatus(orderId=348, permId=20

In [ ]:
logger = OutputLogger()

# # Sync usage
# logger.info("Starting sync task", timed=True)
# logger.warn("This is a warning")
# logger.error("Something failed")

# # Async usage
# async def run_async_stuff():
#     await logger.info_async("Running async task", timed=True)
#     await asyncio.sleep(1)
#     await logger.warn_async("Async warning")
#     await logger.error_async("Async failure", timed=True)

# await run_async_stuff()

In [ ]:
# logger.info("line1")
# logger.warn("line2")
# logger.error("line3\n\nThis is a multi-line error message.\nIt should render correctly in the output area.")

In [ ]:
# import ipywidgets as widgets
# from IPython.display import display, clear_output
# import sys
# from io import StringIO

# Create two DisplayHandle instances
handle1 = DisplayHandle()
handle2 = DisplayHandle()

# # Create a button and associate it with handle1
# button = widgets.Button(description="Click Me")
# # Display the button using handle1
# handle1.display(button)

# def on_button_clicked(b):
#     # Update handle2 (not the one associated with the button)
#     handle2.update("Button was clicked!" + time.strftime("%Y-%m-%d %H:%M:%S"))

# button.on_click(on_button_clicked)


# # Display an initial message in handle2
# handle2.display("Waiting for button click...")

# # Clear any existing output and create a single output area
# clear_output(wait=True)

# # Clean up any existing buttons to avoid duplicate handlers
# try:
#     for widget in [button_a, button_b, button_c, button_d]:
#         widget.close()
# except NameError:
#     pass  # Variables don't exist yet, which is fine

# output_area = widgets.Output()

# redefine the print_to_output function to use handle2
# Save the original print_to_output function
original_print_to_output = print_to_output

# Redefine print_to_output to use handle2
def print_to_output(*args, **kwargs):
    message = time.strftime("%Y-%m-%d %H:%M:%S") + ": " + " ".join(str(arg) for arg in args)
    if kwargs:
        message += " " + " ".join(f"{k}={v}" for k, v in kwargs.items())
    handle2.update(message)

def on_button_a_clicked(b):
    print_to_output("Button A pressed")

def on_button_b_clicked(b):
    # Submit market order
    action = 'BUY'
    market_order = MarketOrder(action, quantity)
    # ib.orderStatusEvent += on_order_status
    trade: Trade = ib.placeOrder(contract, market_order)
    # trade.statusEvent += partial(generic_event_handler, 'tradeStatusEvent')
    # trade.fillEvent += partial(generic_event_handler, 'tradeFillEvent')
    trade.filledEvent += partial(generic_event_handler, 'tradeFilledEvent')
    trade.filledEvent += on_order_filled

def on_button_c_clicked(b):
    # Submit market order
    action = 'SELL'
    market_order = MarketOrder(action, quantity)
    # ib.orderStatusEvent += on_order_status
    trade: Trade = ib.placeOrder(contract, market_order)
    # trade.statusEvent += partial(generic_event_handler, 'tradeStatusEvent')
    # trade.fillEvent += partial(generic_event_handler, 'tradeFillEvent')
    trade.filledEvent += partial(generic_event_handler, 'tradeFilledEvent')
    trade.filledEvent += on_order_filled

def on_button_d_clicked(b):
    # Submit market order
    action = 'SELL'
    limit_order = LimitOrder(action, quantity, 1.11)
    # ib.orderStatusEvent += on_order_status
    trade: Trade = ib.placeOrder(contract, limit_order)
    # trade.statusEvent += partial(generic_event_handler, 'tradeStatusEvent')
    # trade.fillEvent += partial(generic_event_handler, 'tradeFillEvent')
    trade.filledEvent += partial(generic_event_handler, 'tradeFilledEvent')
    trade.filledEvent += on_order_filled

# Create buttons
button_a = widgets.Button(description="Limit Buy")
button_b = widgets.Button(description="Buy")
button_c = widgets.Button(description="Sell")
button_d = widgets.Button(description="Limit Sell")

# Attach event handlers
button_a.on_click(on_button_a_clicked)
button_b.on_click(on_button_b_clicked)
button_c.on_click(on_button_c_clicked)
button_d.on_click(on_button_d_clicked)

# Display the widgets
handle1.display(widgets.HBox([button_a, button_b, button_c, button_d]))
handle2.display("Waiting for button click...")

# display(output_area)

"2025-09-19 13:28:47: Order filled: qty=1.0, price=6707.5; Stop order submitted: StopOrder(orderId=368, clientId=13, action='BUY', totalQuantity=1.0, auxPrice=6708.5)"

Tick data

In [13]:
# for contract in contracts:
ib.reqMktData(contract, '', False, False)

Ticker(contract=Contract(secType=FUT, symbol=ES, lastTradeDateOrContractMonth=202509, exchange=CME))

In [18]:
t = ib.tickers()[0]
data = {
    'symbol': t.contract.symbol,
    'bid': t.bid,
    'ask': t.ask,
    'last': t.last,
    'lastSize': t.lastSize,
    'volume': t.volume,
    'high': t.high,
    'low': t.low,
    'close': t.close,
    'time': t.time.astimezone().strftime('%Y-%m-%d %H:%M:%S.%f') if t.time else None
}
df = pd.DataFrame([data])
display(HTML(df.to_html(index=False)))

symbol,bid,ask,last,lastSize,volume,high,low,close,time
ES,6409.0,6409.25,6409.25,6.0,153971.0,6417.5,6403.25,6406.0,2025-07-30 09:43:11.093843


In [ ]:
# from IPython.display import display, clear_output, update_display

# df = pd.DataFrame(
#     index=[c.conId for c in contracts],
#     columns=['symbol','bidSize', 'bid', 'ask', 'askSize', 'high', 'low', 'close', 'time'])
# Initial display with a display_id
my_display = display("Initial content", display_id="pending_tickers")

def update_display_ticker(ticker, display_id='pending_tickers'):
    # Create a DataFrame from the ticker data
    data = {
        'symbol': ticker.contract.symbol,
        'bidSize': ticker.bidSize,
        'bid': ticker.bid,
        'ask': ticker.ask,
        'askSize': ticker.askSize,
        'high': ticker.high,
        'low': ticker.low,
        'close': ticker.close,
        'time': ticker.time.isoformat() if ticker.time else None
    }
    df = pd.DataFrame([data])
    
    # Update the display with the new DataFrame
    update_display(HTML(df.to_html(index=False)), display_id=display_id)
    # clear_output(wait=True)

def onPendingTickers(tickers):
    for t in tickers:
        # df.loc[t.contract.conId] = (
        #     t.contract.symbol, t.bidSize, t.bid, t.ask, t.askSize, t.high, t.low, t.close, t.time)
        # clear_output(wait=True)
        update_display_ticker(t, display_id='pending_tickers')

ib.pendingTickersEvent += onPendingTickers
# ib.sleep(60)
# ib.pendingTickersEvent -= onPendingTickers

symbol,bidSize,bid,ask,askSize,high,low,close,time
ES,9.0,6363.75,6364.0,11.0,6373.5,6359.5,6374.25,2025-07-31T22:23:52.481720+00:00


In [ ]:
ib.pendingTickersEvent -= onPendingTickers
ib.cancelMktData(contract)

Test `ib.brackerOrder()` BracketOrder

In [ ]:
action = 'BUY'
limitPrice = 6390
takeProfitPrice = limitPrice * 1.0025
stopLossPrice = limitPrice * 0.9995
bracket = ib.bracketOrder(action, quantity,
                 limitPrice=limitPrice, takeProfitPrice=takeProfitPrice, stopLossPrice=stopLossPrice,
                 outsideRth=True, tif='GTC')
trades = []
for o in bracket:
    trade = ib.placeOrder(contract, o)
    trades.append(trade)

print(f"trades: {trades}")

In [ ]:
# quantity = 100
limitPrice = 6390
takeProfitPrice = round(limitPrice * 1.0025 / mintick) * mintick
stopLossPrice = round(limitPrice * 0.9995 / mintick) * mintick
action = 'BUY'
assert action in ('BUY', 'SELL')
reverseAction = 'BUY' if action == 'SELL' else 'SELL'
parent = LimitOrder(
    action, quantity, limitPrice,
    # orderId=ib.client.getReqId(),
    transmit=True,
    )
print(f"parentOrder: {parent}")
# takeProfit = LimitOrder(
#     reverseAction, quantity, takeProfitPrice,
#     orderId=ib.client.getReqId(),
#     transmit=False,
#     parentId=parent.orderId,
#     )
stopLoss = StopOrder(
    reverseAction, quantity, stopLossPrice,
    # orderId=ib.client.getReqId(),
    transmit=True,
    parentId=parent.orderId,
    )
print(f"stopLossOrder: {stopLoss}"
      )
tradeParent = ib.placeOrder(contract, parent)
tradeChild = ib.placeOrder(contract, stopLoss)

Create a stop order

In [ ]:
stopOrder = StopOrder('SELL', 100, 585.02)
stopTrade = ib.placeOrder(contract, stopOrder)
ib.sleep(1)
stopTrade

In [ ]:
ib.sleep(1)
stopTrade

``trade`` contains the order and everything related to it, such as order status, fills and a log.
It will be live updated with every status change or fill of the order.

In [ ]:
ib.sleep(1)
trade.log

``trade`` will also available from ``ib.trades()``:

In [ ]:
assert trade in ib.trades()

Likewise for ``order``:

In [ ]:
assert order in ib.orders()

Now let's create a limit order with an unrealistic limit:

In [ ]:
limitOrder = LimitOrder('BUY', 20000, 0.05)
limitTrade = ib.placeOrder(contract, limitOrder)

limitTrade

``status`` will change from "PendingSubmit" to "Submitted":

In [ ]:
ib.sleep(1)
assert limitTrade.orderStatus.status == 'Submitted'

In [ ]:
assert limitTrade in ib.openTrades()

Let's modify the limit price and resubmit:

In [ ]:
limitOrder.lmtPrice = 0.10

ib.placeOrder(contract, limitOrder)

And now cancel it:

In [ ]:
ib.cancelOrder(limitOrder)

In [ ]:
limitTrade.log

placeOrder is not blocking and will not wait on what happens with the order.
To make the order placement blocking, that is to wait until the order is either
filled or canceled, consider the following:

In [ ]:
%%time
order = MarketOrder('BUY', 100)

trade = ib.placeOrder(contract, order)
while not trade.isDone():
    ib.waitOnUpdate()

What are our positions?

In [ ]:
ib.positions()

In [ ]:
ib.openOrders()

In [ ]:
ib.openTrades()

In [ ]:
ib.trades()

What's the total of commissions paid today?

In [ ]:
ib.fills()

In [ ]:
sum(fill.commissionReport.commission for fill in ib.fills())

whatIfOrder can be used to see the commission and the margin impact of an order without actually sending the order:

In [ ]:
order = MarketOrder('SELL', 20000)
ib.whatIfOrder(contract, order)

In [11]:
ib.disconnect()

In [ ]:
import ib_insync.objects
import datetime

In [ ]:
x = ib_insync.objects.TradeLogEntry(datetime.datetime.now())

In [ ]:
x

In [ ]:
def friendly_str(self):
    friendly_time: datetime.datetime = self.time.isoformat() #strftime('%Y-%m-%d %H:%M:%S')
    return f"TradeLogEntry(time={friendly_time}, status='{self.status}', message='{self.message}', errorCode={self.errorCode})"

ib_insync.objects.TradeLogEntry.__str__ = friendly_str

# Now, printing the object x will use the friendly time format
pprint.pprint(x)